# Lesson 2.3 — experiments: BFS and UCS in a congested warehouse

Run the cells in order. Before each experiment, write down your prediction in the "Your prediction" cell (double-click it to edit), and only then run the code.

The first cell chooses the implementation:

- `"mine"` uses your `task.py`; solve Part B of the task first;
- `"reference"` uses the course's reference implementation. Set `IMPLEMENTATION = "reference"` if you want to run the notebook before finishing the task.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.search import reference as algorithms

The function `run` runs one search on a map and returns the goal node and the list of expanded nodes. The function `summary` prints the number of steps of the route, its cost and the number of expansions. BFS always comes from the course's reference implementation, and UCS from the implementation chosen in the first cell. In the pictures blue cells are expanded, orange cells are the route, and green hatched cells are slow zones.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from introai.search.reference import bfs
from introai.trace import TraceLimitExceeded, record
from introai.world.plot import plot_map
from introai.world.warehouse import RouteProblem, WarehouseMap, render_text

MAPS = Path.cwd().parents[2] / "data" / "maps"


def run(algorithm, warehouse, limit=100_000):
    """Run one search on a map; return the goal node and the expanded nodes."""
    try:
        with record(limit=limit, kinds={"expand"}) as trace:
            node = algorithm(RouteProblem(warehouse))
    except TraceLimitExceeded:
        raise RuntimeError(
            f"{algorithm.__name__} expanded more than {limit} nodes - check task.py "
            '(or set IMPLEMENTATION = "reference" in the first cell)'
        ) from None
    except TypeError as error:
        if "not supported between instances" not in str(error):
            raise
        raise RuntimeError(
            f"{algorithm.__name__} compared two Node objects - put a counter between "
            "the priority and the node on the heap "
            '(or set IMPLEMENTATION = "reference" in the first cell)'
        ) from None
    if node is None:
        raise RuntimeError(
            f"{algorithm.__name__} found no route, but every map here has one - check task.py "
            '(or set IMPLEMENTATION = "reference" in the first cell)'
        )
    return node, [event.data["node"] for event in trace.of_kind("expand")]


def summary(label, node, expanded):
    print(
        f"{label}: {len(node.solution())} steps, cost {node.path_cost:g}, expanded {len(expanded)}"
    )

## Experiment 1: the corridor or the detour

The robot `R` has to reach the dock `D`. Straight ahead runs a corridor full of slow zones `~`, and two ordinary aisles lead around it.

In [ ]:
congested = WarehouseMap.from_file(MAPS / "congested.txt")
print(render_text(congested))

**Your prediction.** Which way will the robot go according to BFS, and which according to UCS? How much will each route cost? Which algorithm will expand more cells?

*My prediction:* …

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
for ax, (label, algorithm) in zip(axes, [("BFS", bfs), ("UCS", algorithms.ucs)]):
    node, expanded = run(algorithm, congested)
    summary(label, node, expanded)
    path = node.states()
    plot_map(congested, path=path, expanded=[n.state for n in expanded], title=label, ax=ax)
fig.tight_layout()

**Think about it.** Compare the result with your prediction. Why did BFS choose the corridor, even though it was much more expensive? How many slow-zone cells does the UCS route cross, and why can they not be avoided?

## Experiment 2: how short a slow zone?

The function `corridor_map(slow)` builds the same map, but with only `slow` slow cells in the corridor, right in front of the dock. The rest of the corridor is ordinary floor.

**Your prediction.** At how many slow cells in the corridor will UCS stop driving along the corridor and choose the detour? Try to work it out before you run the code.

*My prediction:* …

In [ ]:
def corridor_map(slow):
    """congested.txt with only the last `slow` corridor cells, next to the dock, slow."""
    lines = (MAPS / "congested.txt").read_text().splitlines()
    row = lines[3]
    start, end = row.index("R") + 1, row.index("D")
    lines[3] = row[:start] + "." * (end - start - slow) + "~" * slow + row[end:]
    return WarehouseMap.from_text("\n".join(lines))


slow_cells = list(range(18))
bfs_costs, ucs_costs = [], []
print(f"{'slow':>4} {'BFS steps':>9} {'BFS cost':>8} {'UCS steps':>9} {'UCS cost':>8}")
for slow in slow_cells:
    warehouse = corridor_map(slow)
    bfs_node, _ = run(bfs, warehouse)
    ucs_node, _ = run(algorithms.ucs, warehouse)
    bfs_costs.append(bfs_node.path_cost)
    ucs_costs.append(ucs_node.path_cost)
    print(
        f"{slow:>4} {len(bfs_node.solution()):>9} {bfs_node.path_cost:>8g} "
        f"{len(ucs_node.solution()):>9} {ucs_node.path_cost:>8g}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, costs, color in [("BFS", bfs_costs, "#2a78d6"), ("UCS", ucs_costs, "#eb6834")]:
    ax.plot(
        slow_cells,
        costs,
        color=color,
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=label,
    )
    ax.annotate(
        label,
        (slow_cells[-1], costs[-1]),
        textcoords="offset points",
        xytext=(8, 0),
        va="center",
        color="#0b0b0b",
    )
ax.set_title("Cost of the route found, by length of the slow zone", loc="left", color="#0b0b0b")
ax.set_xlabel("slow cells in the corridor", color="#52514e")
ax.set_xticks(range(0, 18, 2))
ax.set_ylabel("route cost", color="#52514e")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
ax.margins(x=0.08)
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
fig.tight_layout()

**Think about it.** Where do the lines part, and why exactly there? How much does the detour cost, and why does its cost rise only at 17 slow cells? Is there a number of slow cells at which both routes cost the same? Which one does UCS choose then, and what does that depend on?

## Experiment 3: where UCS looks

In the picture from experiment 1 UCS also expanded the cells of the storage area on the left, even though the dock is on the right.

**Your prediction.** Which cells does UCS expand before it removes the goal? Can you describe them with a single condition on cost?

*My prediction:* …

In [ ]:
node, expanded = run(algorithms.ucs, congested)
free = [
    (row, col)
    for row in range(congested.rows)
    for col in range(congested.cols)
    if congested.is_free((row, col))
]
print(f"free cells: {len(free)}, expanded by UCS: {len(expanded)}")
print(f"cost of the route: {node.path_cost:g}")
print(f"the most expensive expanded node: {max(n.path_cost for n in expanded):g}")
storage = [n for n in expanded if n.state[1] < congested.start[1]]
print(f"expanded cells in the storage area left of R: {len(storage)}")

**Think about it.** What is the highest cost of an expanded node compared with the cost of the route? Why does UCS have to look into every cell that can be reached more cheaply than the goal, even one in the opposite direction? What information would let it avoid that? The answer comes in lesson 2.4.

## Summary

Go back to the review questions in the lesson. After your own attempt, you can compare your results and conclusions with the answers in `solutions/02_search/03_uniform_cost_search/answers.md`.